In [207]:
import pandas as pd
import numpy as np
from data_util import find_closest_elements


In [208]:
import pandas_market_calendars as mcal
import datetime

nyse = mcal.get_calendar('NYSE')
TOTAL_SECONDS_ONE_YEAR = 365*24*60*60 # total seconds

def get_market_open_close(day_stamp,no_tzinfo=True):
    early = nyse.schedule(start_date=day_stamp, end_date=day_stamp)
    if len(early) == 0:
        raise LookupError("market not open today!")
    market_open = list(early.to_dict()['market_open'].values())[0]
    market_close = list(early.to_dict()['market_close'].values())[0]
    if no_tzinfo:
        return market_open.replace(tzinfo=None),market_close.replace(tzinfo=None)
    else:
        return market_open,market_close

def get_expiry_tstamp(expiry):
    if not isinstance(expiry,str):
        return np.nan
    expiry = datetime.datetime.strptime(expiry,"%Y-%m-%d")
    _,expiry_tstamp = get_market_open_close(expiry)
    return expiry_tstamp.replace(tzinfo=None)

def get_annualized_time_to_expiration(row,expiry_mapper):
    if isinstance(row.expiry,str):
        expiry = row.expiry
    else:
        expiry = row.expiry.strftime("%Y-%m-%d")
    expiry_tstamp = expiry_mapper[expiry]
    sec_to_expiration = (expiry_tstamp-row.tstamp).total_seconds()
    atte = sec_to_expiration/TOTAL_SECONDS_ONE_YEAR
    return atte


In [209]:
df = pd.read_hdf('spx.h5', 'df')

FileNotFoundError: File spx.h5 does not exist

In [3]:
df.head(10)

,date,forward_price,tau,risk_free_rate,is_call,strike_price,option_price,log_moneyness,implied_volatility,delta,time_to_maturity
0,2013-01-02,1459.912086,0.024658,0.001955,-1.0,1255.0,0.100,-0.151241,0.364894,-0.003813,9
1,2013-01-02,1459.912086,0.024658,0.001955,-1.0,1260.0,0.100,-0.147264,0.356226,-0.003900,9
2,2013-01-02,1459.912086,0.024658,0.001955,-1.0,1270.0,0.100,-0.139359,0.338941,-0.004084,9
3,2013-01-02,1459.912086,0.024658,0.001955,-1.0,1275.0,0.125,-0.135430,0.338461,-0.005015,9
4,2013-01-02,1459.912086,0.024658,0.001955,-1.0,1280.0,0.150,-0.131516,0.336568,-0.005953,9
5,2013-01-02,1459.912086,0.024658,0.001955,-1.0,1285.0,0.200,-0.127618,0.339030,-0.007680,9
6,2013-01-02,1459.912086,0.024658,0.001955,-1.0,1290.0,0.225,-0.123734,0.334715,-0.008646,9
7,2013-01-02,1459.912086,0.024658,0.001955,-1.0,1295.0,0.200,-0.119866,0.320733,-0.008082,9
8,2013-01-02,1459.912086,0.024658,0.001955,-1.0,1300.0,0.150,-0.116012,0.300980,-0.006601,9
9,2013-01-02,1459.912086,0.024658,0.001955,-1.0,1305.0,0.175,-0.112173,0.297554,-0.007673,9


['date', 'forward_price', 'tau', 'risk_free_rate', 'is_call', 'strike_price', 'option_price', 'log_moneyness', 'implied_volatility', 'delta', 'time_to_maturity']


In [286]:
cols=['date', 'forward_price', 'tau', 'risk_free_rate', 'is_call', 'strike_price', 'option_price', 'log_moneyness', 'implied_volatility', 'delta', 'time_to_maturity']

def gen_data(dstamp):
    ticker = "SPXW"
    pq_file = f"/mnt/hd1/data/uw-options-cache/SPX/{dstamp}.parquet.gzip"
    df = pd.read_parquet(pq_file)
    df['tstamp_min'] = df.tstamp_sec.apply(lambda x:x.replace(second=0))
    assert(ticker == list(df.underlying_symbol.unique())[0])
    print(df.shape)
    df = df[df.expiry == dstamp]
    print(df.shape)

    expiry_mapper = {x:get_expiry_tstamp(x) for x in df.expiry.unique()}
    df['date']=df.tstamp_min
    df['forward_price']=df.underlying_price
    df['tau']=df.apply(lambda x: get_annualized_time_to_expiration(x,expiry_mapper),axis=1)
    df['risk_free_rate']=0
    df['is_call']=df.option_type.apply(lambda x: 1 if x == 'call' else -1)
    df['strike_price']=df.strike
    df['option_price']=df.price
    df['log_moneyness']= np.log(df.underlying_price/df.strike)
    # df.implied_volatility
    # df.delta
    df['time_to_maturity']=((df.tau*TOTAL_SECONDS_ONE_YEAR)/(60*60*24)).astype(int)
    
    df = df[(df.tau>0)&(df.log_moneyness.notnull())]
    df = df[cols]
    df['is_ref'] = np.random.rand(len(df)) > 0.8
    # https://quant.stackexchange.com/questions/43596/what-is-forward-moneyness-and-how-to-calculate-it
    if False:
        target_ttms = [0]
        target_deltas = [0.5, 0.25, -0.25]
        df_tmp = df.groupby('date').apply(find_closest_elements, 'time_to_maturity', target_ttms, include_groups=False)
        reference_options = df_tmp.groupby(['date','time_to_maturity']).apply(find_closest_elements, 'delta', target_deltas, include_groups=False)
        df['is_ref'] = 0
        df.loc[reference_options.index.get_level_values(-1), 'is_ref'] = 1
    return df

In [287]:
mylist = []
for dstamp in ["2025-12-22","2025-12-23","2025-12-24"]:
    tdf = gen_data(dstamp)
    mylist.append(tdf)
df = pd.concat(mylist)

(898903, 33)
Index(['executed_at', 'underlying_symbol', 'option_chain_id', 'side', 'strike',
       'option_type', 'expiry', 'underlying_price', 'nbbo_bid', 'nbbo_ask',
       'ewma_nbbo_bid', 'ewma_nbbo_ask', 'price', 'size', 'premium', 'volume',
       'open_interest', 'implied_volatility', 'delta', 'theta', 'gamma',
       'vega', 'rho', 'theo', 'sector', 'exchange', 'report_flags', 'canceled',
       'upstream_condition_detail', 'equity_type', 'tstamp', 'tstamp_sec',
       'tstamp_min'],
      dtype='object')
(679945, 33)
(890130, 33)
Index(['executed_at', 'underlying_symbol', 'option_chain_id', 'side', 'strike',
       'option_type', 'expiry', 'underlying_price', 'nbbo_bid', 'nbbo_ask',
       'ewma_nbbo_bid', 'ewma_nbbo_ask', 'price', 'size', 'premium', 'volume',
       'open_interest', 'implied_volatility', 'delta', 'theta', 'gamma',
       'vega', 'rho', 'theo', 'sector', 'exchange', 'report_flags', 'canceled',
       'upstream_condition_detail', 'equity_type', 'tstamp', 'tsta

In [288]:
df.to_hdf(path_or_buf='spx_w_ref.h5', key='df', complevel=9, complib='blosc')

In [289]:
df.head(10)

,date,forward_price,tau,risk_free_rate,is_call,strike_price,option_price,log_moneyness,implied_volatility,delta,time_to_maturity,is_ref
41704,2025-12-22 14:30:00,6865.21,0.000742,0,-1,6710.0,0.19,0.022868,0.353139,-0.008602,0,False
41705,2025-12-22 14:30:00,6865.21,0.000742,0,-1,6700.0,0.14,0.024359,0.360030,-0.006405,0,False
41706,2025-12-22 14:30:00,6865.21,0.000742,0,-1,6710.0,0.19,0.022868,0.353139,-0.008602,0,False
41707,2025-12-22 14:30:00,6865.21,0.000742,0,-1,6700.0,0.14,0.024359,0.360030,-0.006405,0,True
41708,2025-12-22 14:30:00,6865.21,0.000742,0,-1,6730.0,0.22,0.019891,0.318432,-0.010785,0,False
41709,2025-12-22 14:30:00,6865.21,0.000742,0,-1,6725.0,0.17,0.020635,0.318281,-0.008547,0,False
41710,2025-12-22 14:30:00,6865.21,0.000742,0,-1,6730.0,0.22,0.019891,0.318432,-0.010785,0,False
41711,2025-12-22 14:30:00,6865.21,0.000742,0,-1,6725.0,0.17,0.020635,0.318281,-0.008547,0,False
41870,2025-12-22 14:30:00,6865.21,0.000742,0,-1,6740.0,0.30,0.018407,0.310432,-0.014582,0,False
41871,2025-12-22 14:30:00,6865.21,0.000742,0,-1,6730.0,0.20,0.019891,0.314552,-0.010007,0,False
